# Data preprocessing


## Table of contents

1. [**enc2017 corpus**](#enc2017_corpus)
2. [**Estonian Universal Dependencies' EDT corpus**](#estonian_ud_edt_corpus)
3. [**Homonym disambiguation corpus**](#homonym_disambiguation_corpus)
   1. [Gathering the data](#gathering_the_data)

[End](#end)


In [5]:
# Configurations

# Imports
from pathlib import Path

# Paths
DATA_DIR = Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

<a id='enc2017_corpus'></a>


## enc2017 corpus

Data has been gathered from the [corpus](https://github.com/estnltk/eval_experiments_lrec_2020/blob/master/scripts_and_data/enc2017_selection_plain_texts_json.zip) that was used in evaluation experiments reported in LREC 2020 paper "EstNLTK 1.6: Remastered Estonian NLP Pipeline"[^1].

[^1]: Sven Laur, Siim Orasmaa, Dage Särg, Paul Tammo. "EstNLTK 1.6: Remastered Estonian NLP Pipeline". _Proceedings of The 12th Language Resources and Evaluation Conference_. European Language Resources Association: Marseille, France, May 2020, p. 7154-7162


<a id='estonian_ud_edt_corpus'></a>


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import estnltk
import tqdm
import json

# Add the project's root directory to the Python path
sys.path.append(os.path.abspath("../"))

from EstNLTK_model_training.scripts.preprocessing.enc2017.preprocess import (
    Preprocessor as Enc2017Preprocessor,
)

In [4]:
file_paths = np.array(os.listdir(ENC2017_DIRS["raw"] / "_plain_texts_json"))

In [6]:
# Get unique file name prefixes
file_name_prefixes = np.array([file_name.split("_")[0] for file_name in file_paths])
unique_prefixes = np.unique(file_name_prefixes)
print("Unique file name prefixes:")
for prefix in unique_prefixes:
    print(prefix)

Unique file name prefixes:
nc
web13
wiki17


In [ ]:
outer = tqdm.tqdm(
    file_paths,
    desc="Processing files",
    unit="file",
    total=len(file_paths),
)

text_types = list()

for file_name in outer:
    file_path = ENC2017_DIRS["raw"] / "_plain_texts_json" / file_name
    with open(file_path, "r", encoding="utf-8") as f:
        text_json = json.load(f)
        # Extract text type from metadata
        text_type = text_json.get("meta", {}).get("texttype", "unknown")
        text_types.append((file_name, text_type))

Processing files: 100%|██████████| 18486/18486 [00:03<00:00, 5283.04file/s]


In [26]:
text_types_df = pd.DataFrame(text_types, columns=["file_name", "text_type"])

In [27]:
display(text_types_df["text_type"].value_counts())

text_type
unknown        12286
periodicals     5917
science          230
fiction           53
Name: count, dtype: int64

## Estonian Universal Dependencies' EDT [corpus](https://github.com/UniversalDependencies/UD_Estonian-EDT)


<a id='homonym_disambiguation_corpus'></a>


## Homonym disambiguation corpus


In [ ]:
import os
import json
import warnings
import types
import pandas as pd
import numpy as np
import estnltk, estnltk.converters, estnltk.taggers
import sklearn

# from bert_morph_tagger_notebook_functions import NotebookFunctions

# from simpletransformers.ner import NERModel, NERArgs
from tqdm import tqdm
from bert_morph_tagger import BertMorphTagger
from estnltk.converters.label_studio.labelling_configurations import (
    PhraseClassificationConfiguration,
)
from estnltk.converters.label_studio.labelling_tasks import PhraseClassificationTask

e:\Git_projects\EstNLTK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id='gathering_the_data'></a>


### Gathering the data


In [ ]:
# Use specific annotation configurations that were used in homonyms dataset
annotation_confs = {
    1: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg n": "sg n", "sg g": "sg g"},
        header="Vali sõna morfoloogiline vorm (sg n - ainsuse nimetav, sg g -- ainsuse omastav):",
        header_placement="middle",
    ),
    16: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg n": "sg n", "sg g": "sg g"},
        header="Vali sõna morfoloogiline vorm (sg n - ainsuse nimetav, sg g -- ainsuse omastav):",
        header_placement="middle",
    ),
    17: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg n": "sg n", "sg g": "sg g", "sg p": "sg p"},
        header="Vali sõna morfoloogiline vorm (sg n - ainsuse nimetav, sg g - ainsuse omastav, sg p - ainsuse osastav):",
        header_placement="middle",
    ),
    19: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg g": "sg g", "sg p": "sg p", "adt": "adt"},
        header="Vali sõna morfoloogiline vorm (sg g - ainsuse omastav, sg p - ainsuse osastav, adt - lühike sisseütlev):",
        header_placement="middle",
    ),
}

In [ ]:
# Collect input files
input_files = []
input_dir = "./homonymous_word_forms/annotations/"
for fname in os.listdir(input_dir):
    if os.path.isdir(os.path.join(input_dir, fname)):
        for subfname in os.listdir(os.path.join(input_dir, fname)):
            if subfname.endswith(".json"):
                inflection_type = int(subfname.split("_")[2])  # infl_type_xx_1000_vx...
                input_files.append(
                    (
                        inflection_type,
                        "./homonymous_word_forms/annotations/" + fname + "/" + subfname,
                    )
                )

if not input_files:
    raise RuntimeError("No input files found!")

print(f"Found {len(input_files)} input files.")

Found 8 input files.


In [ ]:
overall_data = []
# Extract data from input files
for infl_type, input_file in input_files:
    print(f"Processing file: {input_file} (inflection type {infl_type})")
    num = input_file.split("/")[3]
    annotation_conf = annotation_confs[infl_type]
    with open(input_file, "r", encoding="utf-8") as f:
        raw = f.read()
    task = PhraseClassificationTask(
        annotation_conf,
        input_layer="morph",
        output_layer="morph",
        label_attribute="label",
    )
    classified_sentences = task.import_data(raw)

    # Create dataframe from classified sentences
    data = []
    for sentence in classified_sentences:
        for annotation in sentence.morph:
            if (
                "class_label" in sentence.meta
                and sentence.meta["class_label"] is not None
            ):
                data.append(
                    {
                        "num": num,
                        "inflection_type": infl_type,
                        "sentence": sentence.text,
                        "word": annotation.text,
                        "word_span": (annotation.start, annotation.end),
                        "label": sentence.meta["class_label"],
                    }
                )
    df = pd.DataFrame(data)
    output_csv = (
        f"./homonymous_word_forms/processed/homonyms_infltype_{num}_{infl_type}.csv"
    )
    df.to_csv(output_csv, index=False)
    print(f"Saved processed data to {output_csv}")
    overall_data.extend(data)

# Create overall dataframe
overall_df = pd.DataFrame(overall_data)
overall_output_csv = "./homonymous_word_forms/processed/homonyms_overall.csv"
overall_df.to_csv(overall_output_csv, index=False)
print(f"Saved overall processed data to {overall_output_csv}")

Processing file: ./homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json (inflection type 1)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_1.csv
Processing file: ./homonymous_word_forms/annotations/1/infl_type_16_1000_v1_project-2-at-2024-11-21-19-27-9e8ae0c2.json (inflection type 16)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_16.csv
Processing file: ./homonymous_word_forms/annotations/1/infl_type_17_1000_v1.json (inflection type 17)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_17.csv
Processing file: ./homonymous_word_forms/annotations/1/infl_type_19_1000_v1_project-6-at-2025-11-15-14-13-42753676.json (inflection type 19)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_19.csv
Processing file: ./homonymous_word_forms/annotations/2/infl_type_01_1000_v2_project-5-at-2024-12-11-21-53-280753b4.json (inflection type 

<a id='end'></a>


### END
